# 02. M2 전처리 – raw 데이터 6개 점검
각 셀을 위에서부터 `Shift+Enter`로 한 칸씩 실행하세요. 이 노트북은 파일을 **읽기만** 하고 raw는 수정하지 않습니다.

## 1. raw 파일 찾기

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent          # 노트북이 하위 폴더(notebooks/)에 있을 때
RAW = ROOT / "data" / "raw"
files = sorted(p for p in RAW.glob("*") if p.suffix.lower() in (".csv", ".xlsx", ".xls"))
print("raw 폴더:", RAW)
for i, p in enumerate(files, 1):
    print(i, p.name)

## 2. CSV: 인코딩 자동 판별 + 크기·컬럼·결측·중복

In [ ]:
def read_csv_auto(p):
    for enc in ("utf-8-sig", "cp949"):
        try:
            return pd.read_csv(p, encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"인코딩 판별 실패: {p.name}")

dfs = {}
for p in files:
    if p.suffix.lower() != ".csv":
        continue
    df, enc = read_csv_auto(p)
    df.columns = df.columns.astype(str).str.replace(r"\s+", "", regex=True)
    dfs[p.name] = df
    miss = df.isna().sum()
    print(f"\n== {p.name} | {enc} | {df.shape[0]}행 x {df.shape[1]}열")
    print("컬럼:", list(df.columns))
    print("결측:", miss[miss > 0].to_dict() or "없음", "| 중복 행:", int(df.duplicated().sum()))

## 3. 엑셀: 모든 시트를 열어서 표인지 문서인지 확인
1번 파일은 첫 시트가 비어 있었으므로 시트 전체를 봅니다.

In [ ]:
for p in files:
    if p.suffix.lower() not in (".xlsx", ".xls"):
        continue
    sheets = pd.read_excel(p, sheet_name=None, header=None)
    print(f"\n== {p.name} | 시트 {len(sheets)}개")
    for name, df in sheets.items():
        filled = int(df.notna().sum().sum())
        print(f"  [{name}] {df.shape[0]}행 x {df.shape[1]}열, 값 있는 칸 {filled}개")
        for v in df.stack().head(6):
            print("     ·", str(v)[:90].replace("\n", " "))

## 4. 요약표 (노션에 붙여넣기용)

In [ ]:
rows = []
for name, df in dfs.items():
    num = df.select_dtypes("number")
    out = 0
    for c in num.columns:
        q1, q3 = num[c].quantile([0.25, 0.75]); iqr = q3 - q1
        out += int(((num[c] < q1 - 1.5*iqr) | (num[c] > q3 + 1.5*iqr)).sum())
    rows.append([name, df.shape[0], df.shape[1], int(df.isna().sum().sum()), int(df.duplicated().sum()), out])
summary = pd.DataFrame(rows, columns=["파일", "행", "열", "결측합", "중복", "이상치후보(IQR)"])
summary